# Support Vector Machines (SVMs)

## Learning Objectives
1. Implement hard-margin SVM geometry — support vectors and maximum margin
2. Apply SVC with multiple kernels (linear, RBF, poly, sigmoid); tune C and gamma
3. Use LinearSVC for high-dimensional text classification with TF-IDF features
4. Visualise the kernel trick in 2D → 3D and compare SVM vs Logistic Regression


In [ ]:
import numpy as np
import torch
import torch.nn as nn
from sklearn.datasets import make_classification, make_circles
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import time

np.random.seed(42)
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


## Level 1 — Hard-Margin SVM Geometry from Scratch

In [ ]:
# ── Level 1: SVM margin and support vectors (numpy) ──────────────────────
import numpy as np

np.random.seed(42)

# Linearly separable 2D data
X_pos = np.array([[1, 2], [2, 3], [3, 3], [2, 1]])    # class +1
X_neg = np.array([[-1, -1], [-2, -2], [-1, -3], [-3, -2]])  # class -1
X = np.vstack([X_pos, X_neg]).astype(float)
y = np.array([1, 1, 1, 1, -1, -1, -1, -1], dtype=float)

# Analytical hard-margin SVM for 2-class linearly separable 2D data
# Use sklearn for exact solution, then extract geometry
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_sc   = scaler.fit_transform(X)

svm = SVC(kernel="linear", C=1e6)   # large C ≈ hard margin
svm.fit(X_sc, y)

w_vec  = svm.coef_[0]               # normal to decision boundary
b_val  = svm.intercept_[0]
sv_idx = svm.support_               # indices of support vectors

margin = 2 / np.linalg.norm(w_vec)

print("Hard-margin SVM geometry:")
print(f"  Decision boundary: {w_vec[0]:.4f}*x1 + {w_vec[1]:.4f}*x2 + {b_val:.4f} = 0")
print(f"  Margin width:      {margin:.4f}  (maximised to separate classes)")
print(f"  Support vectors ({len(sv_idx)}):")
for i in sv_idx:
    print(f"    X[{i}] = {X[i]}  y={y[i]:+.0f}  "
          f"functional margin = {y[i]*(X_sc[i] @ w_vec + b_val):.4f}  "
          f"(should be ≈ 1)")

print("\nOnly the support vectors determine the boundary; other points are irrelevant.")


## Level 2 — SVC with Multiple Kernels and Grid Search

In [ ]:
# ── Level 2: kernel comparison and hyperparameter tuning ─────────────────
import numpy as np, time
from sklearn.datasets import make_circles, make_classification
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

np.random.seed(42)

# Non-linearly separable data (nested circles)
X_nl, y_nl = make_circles(n_samples=400, noise=0.1, factor=0.4, random_state=42)
X_tr_nl, X_te_nl, y_tr_nl, y_te_nl = train_test_split(
    X_nl, y_nl, test_size=0.25, random_state=42)

kernels = ["linear", "rbf", "poly", "sigmoid"]
print("Kernel comparison on non-linearly separable data (nested circles):")
print(f"{'Kernel':<10}  {'Test Acc':>8}  {'Train ms':>10}")
print("-" * 35)
for k in kernels:
    pipe = Pipeline([("sc", StandardScaler()),
                     ("svm", SVC(kernel=k, C=1.0, gamma="scale", degree=3))])
    t0   = time.perf_counter()
    pipe.fit(X_tr_nl, y_tr_nl)
    tms  = (time.perf_counter() - t0) * 1000
    acc  = pipe.score(X_te_nl, y_te_nl)
    print(f"{k:<10}  {acc:>8.3f}  {tms:>10.1f}")

# Grid search for best C and gamma (RBF)
print("\nGrid search: C × gamma for RBF kernel")
param_grid = {"svm__C": [0.1, 1.0, 10.0], "svm__gamma": ["scale", "auto", 0.01, 0.1]}
pipe_rbf = Pipeline([("sc", StandardScaler()), ("svm", SVC(kernel="rbf"))])
gs = GridSearchCV(pipe_rbf, param_grid, cv=5, scoring="accuracy", n_jobs=-1)
gs.fit(X_tr_nl, y_tr_nl)
print(f"Best params: {gs.best_params_}  CV acc: {gs.best_score_:.3f}")
print(f"Test acc with best params: {gs.score(X_te_nl, y_te_nl):.3f}")


## Real-World Example 1 — SVM for Text Classification (TF-IDF + LinearSVC)

In [ ]:
# ── RW1: high-dimensional text classification with LinearSVC ─────────────
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import time

np.random.seed(42)

# Synthetic "documents": two topics (tech vs sports)
tech_words   = ["model", "network", "data", "algorithm", "training",
                "tensor", "gradient", "layer", "feature", "embedding"]
sports_words = ["goal", "score", "team", "player", "match",
                "tournament", "referee", "ball", "stadium", "league"]

def make_doc(vocab, n_words=30):
    return " ".join(np.random.choice(vocab, n_words))

docs   = [make_doc(tech_words) for _ in range(600)] +          [make_doc(sports_words) for _ in range(600)]
labels = [0] * 600 + [1] * 600
np.random.shuffle(combined := list(zip(docs, labels)))
docs, labels = zip(*combined)

X_tr, X_te, y_tr, y_te = train_test_split(docs, labels, test_size=0.2, random_state=42)

# TF-IDF vectorisation (creates high-dimensional sparse matrix)
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_tr_vec = tfidf.fit_transform(X_tr)
X_te_vec  = tfidf.transform(X_te)
print(f"TF-IDF feature matrix: {X_tr_vec.shape}  "
      f"(sparse: {100 * (1 - X_tr_vec.nnz / (X_tr_vec.shape[0] * X_tr_vec.shape[1])):.1f}% zeros)")

# LinearSVC vs LogisticRegression in high-dimensional space
models = {"LinearSVC": LinearSVC(C=1.0, max_iter=2000, random_state=42),
          "LogRegL2":  LogisticRegression(C=1.0, solver="lbfgs", max_iter=500)}

for name, clf in models.items():
    t0  = time.perf_counter()
    clf.fit(X_tr_vec, y_tr)
    tms = (time.perf_counter() - t0) * 1000
    acc = clf.score(X_te_vec, y_te)
    print(f"{name:<12}: acc={acc:.3f}  train_ms={tms:.1f}")

print("\nLinearSVC is faster in high-dimensional sparse text spaces due to primal formulation.")


## Real-World Example 2 — Kernel Trick Visualisation (2D → 3D)

In [ ]:
# ── RW2: visualise RBF kernel lifting non-separable data into 3D ─────────
import numpy as np
from sklearn.datasets import make_circles
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
X2d, y = make_circles(n_samples=200, noise=0.05, factor=0.4, random_state=42)

# RBF feature map approximation (Nystroem) — explicitly shows the higher-dim mapping
from sklearn.kernel_approximation import Nystroem
nystroem = Nystroem(kernel="rbf", gamma=1.0, n_components=3, random_state=42)
X3d = nystroem.fit_transform(X2d)

print("Kernel trick demonstration:")
print(f"  Input space: {X2d.shape}  →  Lifted space: {X3d.shape}")

# Check separability in 2D vs lifted 3D
from sklearn.svm import LinearSVC
from sklearn.preprocessing import StandardScaler

sc2d = StandardScaler(); sc3d = StandardScaler()
X2d_sc = sc2d.fit_transform(X2d)
X3d_sc = sc3d.fit_transform(X3d)

svm2d = LinearSVC(max_iter=2000, random_state=42).fit(X2d_sc, y)
svm3d = LinearSVC(max_iter=2000, random_state=42).fit(X3d_sc, y)

acc2d = svm2d.score(X2d_sc, y)
acc3d = svm3d.score(X3d_sc, y)

print(f"  Linear SVM in 2D (original):   accuracy = {acc2d:.3f}  (non-separable)")
print(f"  Linear SVM in 3D (RBF-lifted): accuracy = {acc3d:.3f}  (separable!)")
print("\nThe kernel trick implicitly computes the inner product in the high-dim space.")
print("RBF SVC achieves the same result without explicitly constructing X3d.")

# Confirm with RBF SVC
rbf_svc = SVC(kernel="rbf", gamma=1.0, C=10)
rbf_svc.fit(X2d_sc, y)
print(f"  RBF SVC on 2D (implicit):       accuracy = {rbf_svc.score(X2d_sc, y):.3f}")
print(f"  Support vectors used:           {len(rbf_svc.support_)}")


## Real-World Example 3 — SVM vs Logistic Regression on High-Dimensional Data

In [ ]:
# ── RW3: SVM margin advantage with many features ─────────────────────────
import numpy as np, time
from sklearn.datasets import make_classification
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

np.random.seed(42)

print(f"{'n_features':>12}  {'LinearSVC CV':>14}  {'LogReg CV':>12}  {'SVC Time (s)':>14}")
print("-" * 58)

for n_feat in [20, 100, 500, 2000]:
    X, y = make_classification(n_samples=500, n_features=n_feat,
                               n_informative=max(5, n_feat // 10),
                               n_redundant=max(2, n_feat // 20),
                               random_state=42)

    pipe_svm = Pipeline([("sc", StandardScaler()),
                          ("clf", LinearSVC(C=1.0, max_iter=3000))])
    pipe_lr  = Pipeline([("sc", StandardScaler()),
                          ("clf", LogisticRegression(C=1.0, solver="lbfgs",
                                                      max_iter=500))])

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    t0 = time.perf_counter()
    svm_scores = cross_val_score(pipe_svm, X, y, cv=cv, scoring="accuracy")
    svm_time   = time.perf_counter() - t0
    lr_scores  = cross_val_score(pipe_lr,  X, y, cv=cv, scoring="accuracy")

    print(f"{n_feat:>12}  {svm_scores.mean():>12.3f}±{svm_scores.std():.3f}  "
          f"{lr_scores.mean():>10.3f}±{lr_scores.std():.3f}  {svm_time:>14.2f}")

print("\nLinearSVC tends to generalise better with many features due to max-margin principle.")
